> **Credentials:** set `KAGGLE_USERNAME`, `KAGGLE_KEY`, and `WANDB_API_KEY` in the environment.
> Never hardcode credentials in this notebook.


# TR_ABDOMEN_RAD staged training (wandb-monitored)

One session = 1 epoch of YOLO26-L-P2 @1280 on 2xT4, validated on the full val split each epoch.
Training + val curves go to Weights & Biases; the test split is touched only in the final evaluation.

Each session downloads the `huseyincavus/abdomen-ckpt` dataset via API (resume if present), trains, validates, then uploads the new `last.pt` as a new dataset version.
Fire the next session with `WANDB_API_KEY=... python trainer/push_staged.py` (credentials are injected at push time).

In [ ]:
import os
import torch

print("torch", torch.__version__)
print("cuda", torch.cuda.is_available(), torch.cuda.device_count())
for i in range(torch.cuda.device_count()):
    print(" ", i, torch.cuda.get_device_name(i))
print("input:", sorted(os.listdir("/kaggle/input")))

In [ ]:
!pip -q install ultralytics==8.4.93 kagglehub kaggle wandb scikit-learn

In [ ]:
import json
import os
from pathlib import Path

import wandb

KAGGLE_USERNAME = os.environ.get("KAGGLE_USERNAME", "")
KAGGLE_KEY = os.environ.get("KAGGLE_KEY", "")
if not (KAGGLE_USERNAME and KAGGLE_KEY):
    raise SystemExit("Set KAGGLE_USERNAME and KAGGLE_KEY in the environment")

kaggle_path = Path.home() / ".kaggle" / "kaggle.json"
kaggle_path.parent.mkdir(parents=True, exist_ok=True)
kaggle_path.write_text(json.dumps({"username": KAGGLE_USERNAME, "key": KAGGLE_KEY}))
kaggle_path.chmod(0o600)

wandb_key = os.environ.get("WANDB_API_KEY")
wandb.login(key=wandb_key, relogin=True) if wandb_key else wandb.login()


In [ ]:
import os

def pick(paths):
    for p in paths:
        if os.path.isdir(p):
            return p
    return None

PREPARED = pick([
    "/kaggle/input/abdomen-prepared/yolo_dataset/images",
    "/kaggle/input/datasets/huseyincavus/abdomen-prepared/yolo_dataset/images",
])
if PREPARED is None:
    raise FileNotFoundError("abdomen-prepared images not mounted")

LABELS = pick([
    "/kaggle/input/abdomen-yolo-labels/labels",
    "/kaggle/input/datasets/huseyincavus/abdomen-yolo-labels/labels",
])
if LABELS is None:
    from kaggle.api.kaggle_api_extended import KaggleApi
    api = KaggleApi()
    api.authenticate()
    api.dataset_download_files("huseyincavus/abdomen-yolo-labels", path="/kaggle/working/labels_dl", unzip=True, quiet=True)
    LABELS = "/kaggle/working/labels_dl/labels"
print("prepared:", PREPARED)
print("labels:", LABELS)

In [ ]:
import os
import shutil

code_root = None
for d in (
    "/kaggle/input/tr-abdomen-rad-code",
    "/kaggle/input/datasets/axqetc/tr-abdomen-rad-code",
    "/kaggle/input/datasets/huseyincavus/tr-abdomen-rad-code",
):
    if os.path.isdir(d):
        code_root = d
        break
if code_root is None:
    from kaggle.api.kaggle_api_extended import KaggleApi
    api = KaggleApi()
    api.authenticate()
    api.dataset_download_files("axqetc/tr-abdomen-rad-code", path="/kaggle/working/code_dl", unzip=True, quiet=True)
    code_root = "/kaggle/working/code_dl"
print("code_root:", code_root)
for d in ("src", "configs", "trainer", "annotations"):
    shutil.copytree(os.path.join(code_root, d), f"/kaggle/working/{d}", dirs_exist_ok=True)
print(sorted(os.listdir("/kaggle/working")))

In [ ]:
from pathlib import Path

DS = Path("/kaggle/working/ds")
for split in ("train", "val", "test"):
    for kind, src in (("images", PREPARED), ("labels", LABELS)):
        dst = DS / kind / split
        dst.parent.mkdir(parents=True, exist_ok=True)
        if not dst.exists():
            dst.symlink_to(f"{src}/{split}")
(DS / "dataset.yaml").write_text(
    f"path: {DS}\n"
    "train: images/train\nval: images/val\ntest: images/test\n"
    "nc: 6\nnames: ['acute_appendicitis','acute_cholecystitis','acute_pancreatitis','kidney_ureteral_stone','acute_diverticulitis','aaa_dissection']\n"
)
print((DS / "dataset.yaml").read_text())
print("train imgs:", len(list((DS / "images" / "train").glob("*.jpg"))))

In [ ]:
import os
from pathlib import Path

from ultralytics import YOLO

TARGET_EPOCHS = 1
CKPT = None
for d in ("/kaggle/input/axqetc/abdomen-ckpt", "/kaggle/input/datasets/axqetc/abdomen-ckpt"):
    if os.path.isdir(d):
        for root, _, files in os.walk(d):
            if "last.pt" in files:
                CKPT = os.path.join(root, "last.pt")
                break
    if CKPT:
        break
if CKPT is None:
    try:
        from kaggle.api.kaggle_api_extended import KaggleApi
        api = KaggleApi()
        api.authenticate()
        api.dataset_download_files("axqetc/abdomen-ckpt", path="/kaggle/working/ckpt", unzip=True, quiet=True)
        for root, _, files in os.walk("/kaggle/working/ckpt"):
            if "last.pt" in files:
                CKPT = os.path.join(root, "last.pt")
                break
    except Exception as e:
        print("no checkpoint dataset yet:", str(e)[:120])

def stop_after_epoch(trainer):
    trainer._session_epochs = getattr(trainer, "_session_epochs", 0) + 1
    if trainer._session_epochs >= TARGET_EPOCHS:
        trainer.stop = True

if CKPT:
    print("resuming from", CKPT, flush=True)
    model = YOLO(CKPT)
    model.add_callback("on_train_epoch_end", stop_after_epoch)
    model.train(resume=True)
else:
    model = YOLO("/kaggle/working/configs/yolo26l-p2.yaml")
    model.add_callback("on_train_epoch_end", stop_after_epoch)
    model.train(
        data="/kaggle/working/ds/dataset.yaml",
        pretrained="yolo26l.pt",
        epochs=25, imgsz=1280, batch=8, device=[0, 1],
        optimizer="AdamW", lr0=0.0007, cos_lr=True,
        mosaic=1.0, close_mosaic=10, fliplr=0.0,
        save_period=1, project="tr-abdomen-rad", name="yolo26l-p2",
        seed=0, workers=2, cache=False, patience=100, exist_ok=True,
    )
print("training done", flush=True)

In [ ]:
import json
import shutil
from pathlib import Path
from kaggle.api.kaggle_api_extended import KaggleApi

LAST = Path("/kaggle/working/tr-abdomen-rad/yolo26l-p2/weights/last.pt")
stage = Path("/kaggle/working/ckpt_upload")
if LAST.exists():
    if stage.exists():
        shutil.rmtree(stage)
    stage.mkdir(parents=True)
    shutil.copy2(LAST, stage / "last.pt")
    (stage / "dataset-metadata.json").write_text(json.dumps({
        "title": "abdomen_ckpt",
        "id": "axqetc/abdomen-ckpt",
        "licenses": [{"name": "CC0-1.0"}],
        "isPrivate": True,
        "description": "Staged training checkpoint for TR_ABDOMEN_RAD.",
    }, indent=2))
    api = KaggleApi()
    api.authenticate()
    try:
        r = api.dataset_create_version(str(stage), version_notes="staged checkpoint", dir_mode="zip", quiet=True)
    except Exception:
        r = api.dataset_create_new(str(stage), public=False, quiet=True, dir_mode="zip")
    print("ckpt upload:", getattr(r, "status", None), flush=True)
else:
    print("no last.pt to upload", flush=True)

In [ ]:
from pathlib import Path

p = Path("/kaggle/working/tr-abdomen-rad/yolo26l-p2/results.csv")
if p.exists():
    print(p.read_text())
w = Path("/kaggle/working/tr-abdomen-rad/yolo26l-p2/weights")
print(sorted(x.name for x in w.iterdir()) if w.exists() else "no weights")